<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
</head>
<body>
    <div style="display: flex; align-items: center;">
        <div>
            <h1>TD 3 - Signal Detection Theory and Bayesian Inference</h1>
            <h2>Understanding human behavior with cognitive models</h2>
            <h3>Master in Cognitive Science</h3>
            <h4>École Normale Supérieure - PSL</h4>
            <p> Valentin Wyart - Lecturer<br>
                Amric Trudel - Practical Sessions (TD)<br>
                <a href="mailto:amric.trudel@ens.psl.eu">amric.trudel@ens.psl.eu</a></p>
        </div>
        <div>
            <img src="images/logo_ens.png" style="height: 70px; margin-left: 10px;" />
        </div>
    </div>
</body>
</html>

# Objectives
The objective of this TD is twofold:

1- Get familiar with model comparison/falsification with the Stimulus Detection Task
- Implement the High-threshold theory (HTT) model
- Implement the Signal Detection Theory (SDT) model
- Use the Receiver Operating Characteristic (ROC) curve to display the models

2- Understand the basic principles behind bayesian inference
- Concepts of prior, likelihood, and posterior
- Generative model
- Bayes' rule
- Bayesian inference

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import math
import scipy

%reload_ext autoreload
%autoreload 2

# 1- Signal Detection Theory

## Task Description
We will work on a **stimulus detection task**. In such a task, participants are asked to detect the presence of a signal in a series of trials. The signal is embedded in noise, and the participant must decide whether they think the signal is present or not.
- If the signal is present:
    - if the participant detects a signal ("Yes"): **Hit** ✅
    - if the participant doesn't detect it ("No"): **Miss** ❌
- If the signal is NOT present:
    - if the participant detects a signal ("Yes"): **False Alarm** ❌
    - if the participant doesn't detect it ("No"): **Correct Rejection** ✅

<img src="images/stimulus_detection_task.png" style="height: 300px; margin-left: 10px;" />

## Data loading
Here we load the data from `data.csv`.
In the dataset, each row corresponds to a trial. Data is collected for two participants, and each participant completes 3 blocks of trials. The stimuli and responses are coded as follows:
- 1 codes for the presence of signal in the 'stim' column, and for a 'Yes' response in the 'resp' column
- 0 codes for only noise in the 'stim' column, and for a 'No' response in the 'resp' column.

In [ ]:
data = ...

## Data visualization
To analyze the data, we will build a contingency matrix like the one illustrated in the figure below to display the hits (H), misses (M), false alarms (FA) and correct rejections (CR).

These statistics are computed on **ONE participant**, and **ONE block** at a time.

<img src="images/contingency_table.png" style="height: 200px; margin-left: 10px;" />

### Contingency statistics
📝 Complete the `contingency_statistics` function, which takes in a DataFrame that contains a 'stim' and a 'resp' column **of the same participant and same block**, and returns the hit, false-alarm, miss and correct-rejection rates. A 3-step structure is proposed to write this function:
1. Extract the stimuli and responses from the dataframe
2. Count the number of hits, false alarms, misses and correct rejections in the block
3. Compute the **rates**. For this, you have to normalize the scores so that the sum of each column of the contingency table is equal to 1.

In [ ]:
def contingency_statistics(data_block: pd.DataFrame):
    # Step 1
    stimuli = ...
    responses = ...

    # Step 2
    hits = ...
    fa = ...
    miss = ...
    cr = ...

    # Step 3
    hit_rate = ...
    fa_rate = ...
    miss_rate = ...
    cr_rate = ...

    return hit_rate, fa_rate, miss_rate, cr_rate

📝Call your function with the **first block** of the **first participant**.

You should obtain hit_rate=0.61, fa_rate=0.12, miss_rate=0.39, cr_rate=0.88.

In [ ]:
# Your code here


### Contingency matrix
Hit, miss, false alarm and correction rejection rates are usually visualized in a matrix form.

📝Fill the `plot_contingency_matrix` function to visualize your rates in a contingency matrix like the one shown in the figure above.
Hints:
- You can use the Seaborn library to make a *heatmap*.
- Show the rates as numbers in the cells of the matrix.
- Add a color scale to the side of the matrix.
- Make sure the x and y axes are labeled

In [ ]:
def plot_contingency_matrix(hit_rate, fa_rate, miss_rate, cr_rate):
    # Your code here
    ...


# We then call the function
plot_contingency_matrix(..., ..., ..., ...)

### Plot false alarm rate vs hit rate

📝 In a scatter plot, show the false alarm rate and the hit rate of the block on a scatter plot:
- Place the false alarm rate (FAR) on the x axis, and the hit rate (HR) on the y axis.
- Set the x and y axis limits to (0, 1)
- Even though we only have one hit rate and one false alarm rate to plot, we will write a function to plot multiple points later.
- Don't call `plt.plot()` at the end of your function, as we will add stuff to your graph later

In [ ]:
def plot_hr_far(hit_rates: list, fa_rates: list):
    # Your code here
    ...


# Here we place the HR and FAR of the first block in lists and call the plotting function
plot_hr_far([hit_rate], [fa_rate])

## Theory #1: High-threshold theory
Now it is time to model the participant's behavior.
Before using the signal detection theory, we will first start with the theory that was popular just before: **high-threshold theory**. This theory characterizes each participant by considering that he/she has a distinct **detection sensitivity**. This means that we assume that a signal $s$ will trigger an (unobserved) sensory response $r(s)$. Then, if the response is higher than the detection sensitivity $p$ ($r(s) > p$), the participant will report that they detected a stimulus ("Go"). HTT gives the following formula to estimate $p$ as a function of the hit rate (HR) and the false alarm rate (FAR):
- **Detection sensitivity**: $p = \frac{\text{HR} - \text{FAR}}{1-\text{FAR}}$

And what if the stimulus response is lower than the sensitivity threshold $p$? Then we assume that the participant guesses and reports the presence of a stimulus at a rate given by the **detection criterion**, which is equivalent to the false alarm rate (FAR):
- **Detection criterion**: $g = \text{FAR}$

### Compute detection sensitivity and criterion (HTT)
📝 Fill the `compute_htt` function, which computes $p$ and $g$ based on the hit rate and false alarm rate of a participant.

In [ ]:
def compute_htt(hit_rate: float, fa_rate: float):
    p = ... # Fill this
    g = ... # Fill this
    return p, g

p, g = compute_htt(..., ...) # Fill the arguments

### ROC Curve (HTT)
The purpose of the ROC curve is to visualize the performance of a binary classifier. It is a plot of the hit rate (HR) against the false alarm rate (FAR) for the different possible decision thresholds of the classifier. The ROC curve is a useful tool to compare the performance of different models.

📝 With algebraic manipulations on the HTT equations, find the linear equation that expresses the Hit rate as a function of the False Alarm rate and the detection sensitivity $p$.

$$
\begin{align}
\text{HR}   &= f_p(\text{FAR}) \\
            &= ... \text{fill this}
\end{align}
$$

📝Fill the `plot_htt_roc_curve` function to display the ROC curve on top of the hr_far scatter plot that you made previously.
- Create a vector of 100 false alarm rate values ranging from 0 to 1 (you can use `np.linspace`)
- Calculate the associated hit rates using the equation you just found
- Plot the ROC curve on top of the scatter plot (that was created before calling the function)
- Add a label and legend to specify the model associated with the ROC curve

In [ ]:
def plot_htt_roc_curve(p):
    # Your code here
    ...


# We call the function that generates the scatter plot
plot_hr_far([hit_rate], [fa_rate])
# We then call the function that generates the ROC curve
plot_htt_roc_curve(...) # Fill the arguments

💭Notice here that the ONLY parameter you need to generate the ROC curve is the sensitivity $p$. How is it that the `plot_htt_roc_curve` function doesn't need the criterion $g$ as well?

Answer this question before moving on.


_Your answer here_

## Theory #2: Signal Detection Theory
Signal Detection Theory has replaced High-Threshold Theory as a reference model for stimulus detection. This theory also defines parameters for detection **sensitivity** and **criterion**.
- **Detection sensitivity**: $d' = z(\text{HR}) - z(\text{FAR})$ 
- **Detection criterion**: $c = - \frac{z(\text{HR}) + z(\text{FAR})}{2}$  
  where $z$ is the z-score function (see `scipy.stats.norm.ppf`)


### Compute detection sensitivity and criterion (SDT)
📝 Fill the `compute_sdt` function, which computes $d'$ and $c$ based on the hit rate and false alarm rate of a participant on a block.

In [ ]:
def compute_sdt(hit_rate, fa_rate):
    d_prime = ...   # Fill this
    c = ...         # Fill this
    return d_prime, c


d_prime, c = compute_sdt(..., ...)  # Fill the arguments

### ROC Curve (SDT)
Now we will do the same as we did for the HTT model.

📝 With algebraic manipulations on the SDT equations, find the equation that expresses the Hit rate as a function of the False Alarm rate and the detection sensitivity $d'$.

$$
\begin{align}
\text{HR}   &=  f_{d'}(\text{FAR})  \\
            &=  ...
\end{align}
$$

📝 Fill the `plot_sdt_roc_curve` function to display the ROC curve on top of the hr_far scatter plot that you made previously.
- As you did in the previous ROC curve plotting function, generate a vector of 100 HR values and compute the associated FAR values with the equation you just derived.
- You will need to find the inverse of the z-score function, which exists in scipy
- Add a label and legend to specify the model associated with the ROC curve

In [ ]:
def plot_sdt_roc_curve(d_prime):
    # Your code here
    ...


# Fill the arguments
plot_hr_far(..., ...)
plot_sdt_roc_curve(...)

## Model comparison
We generally use ROC curves to compare the models on the same plot.

📝 Plot both ROC Curves (HT and SDT) on the same graph to compare them.
- Generate the scatter plot of the hit rate and false alarm rate
- Add the ROC curve of the HTT model
- Add the ROC curve of the SDT model

In [ ]:
# Your code here


💭You have one data point, which corresponds to one block of data on the same task difficulty. If you have implemented the models correctly, both curves should be able to fit that point correctly. How would you describe the difference in the predictions that the two models make, and what additional data would you need to falsify one of the models?

**Write an answer to this question before moving to the next section.**

_Your answer here_

## Model falsification
The dataset contains data from 3 different blocks for the first participant. The two alternative models make different predictions about the participant's behavior they change their criterion. Let's see what the actual data looks like on the other blocks that were intended to make the participant more or less conservative by changing the punishment for false alarms.

We can compute the hit rate and the false alarm rates each of the three blocks. We will keep the ROC curves calculated on block 1 and see if its predictions generalize to the other blocks.

📝Create a new plot that allows you to compare the two models on the data from participant 1, and block 1-2-3. Your plot should contain:
- A scatter plot of the hit rates and false alarm rates of **all three blocks**.
- The ROC curve for the HTT model fitted on the data for **block 1 only** (same as you did before)
- The ROC curve for the SDT model fitted on the data for **block 1 only** (same as you did before)

_Hint:_ There are many ways you can achieve this. The cleanest way would be to write a custom function that does all this, but you can also reuse the code from last question and just add the data points from blocks 2 and 3.

In [ ]:
hit_rates_per_block = []
fa_rates_per_block = []
# Compute the hit rates and false alarm rates for each block
...

# Plot the data points for all blocks
plot_hr_far(hit_rates_per_block, fa_rates_per_block)
# Plot the ROC curves for both models (fitted on the first block)
...


💭Can you falsify one of the models with this new data?

# Optional exercises
This part is for people who finish the TD early and want to go further.   

**First complete Part 2 (Bayesian inference) before you start these optional exercises.**

## 💪 Optional exercise 1: Modelling different subjects
So far we have modelled participant 1. Let's see if we can also model participant 2, who has a **higher sensitivity**.

💭 What do you expect to be different between with this participant?
- How do you expect both ROC curve to look like?
- What parameters will change and how?

📝 On the same graph and for both participants, plot:
- Their HR and FAR for all three blocks
- The ROC curve for the HTT model fitted on the first block
- The ROC corve for the SDT model fitted on the first block

In [ ]:
# Your code here



## 💪💪💪 Optional exercise 2: Fitting the SDT model on all three blocks at once
So far, we fitted the $d'$ parameter on the first block only and looked if its predictions could generalize to the other blocks. But how would you go about fitting the model to all three blocks?

📝 Focusing on **participant 1**, find a way to fit the SDT model to the data from blocks 1-2-3 simlultaneously and plot your ROC curve to see if it better approaches all three data points instead of only going through the point from block one.  
_Hint:_ Write down all the steps you have to do before you start writing your code. Don't hesitate to discuss it with me. This exercise is intended to be more difficult and less guided.

In [ ]:
# Your code here


# 2- Bayesian Inference
In this second part of the TD, we will introduce the basic principles behind Bayesian inference, namely **Bayes' Rule**.
Bayes' rule is crucial for understanding perception, and we will use it throughout the course. You may or may not have studied bayesian probability theory before, so this is the occasion to develop your intuition about these very profound concepts.

In this section, we will work on a very simple example from the book [*Bayesian Models of Perception and Action*](https://www.cns.nyu.edu/malab/bayesianbook.html)$^{(1)}$. Imagine that you are walking in the school's corridor and you see a shiny floor. Will you assume that it is wet or that it is just a mere reflection? Will you instinctively start walking cautiously when you see it?

To analyze the situation, we need to break it down into its various possiblities:
- We consider that the floor can have two possible states: **wet** or **dry**
- We can make two possible observations when looking at the floor: it can be **shiny** or **not shiny**.

The figure below shows a shiny floor, and how this observation can transform our prior belief about the usual wetness of that floor, into an updated belief, considering what we observe. We will break down the notions of **prior**, **posterior** and **likelihood** in this exercise.

<img src="images/bayesian_example.png" style="height: 200px; margin-left: 10px;" />

## Step 1: Generative model

The first step of Bayesian modeling is the **generative model**, which represents the statistical structure of the world and the observations. As mentioned above, we consider in our example two possible world states, each of which can generate two possible observations. In the generative model, we need to specify the probability distributions of all variables of the problem.

From our previous experience as an observer, we assume the following:
- **Base rate:** In general, the floor has a 10% probability of being wet.
- **Conditional probability:** If the floor is wet, it has a 80% probabililty of being shiny.
- **Conditional probability:** If the floor is dry, it has a 40% probability of being shiny.

The figure below illustrates the generative model. The conditional probabilities are left for you to deduce from the above statement.

<img src="images/generative_model.png" style="height: 200px; margin-left: 10px;" />

📝 Define the `priors` of each state of the world (regardless of the observation):
- $p(\text{wet})$
- $p(\text{dry})$

In [ ]:
priors = {
    'wet': ...,
    'dry': ...
}

📝 Define the `likelihoods` of each observation, under each hypothesis. They correspond to the conditional probabilities. The values that you need to enter in this dictionary are, in order:
- $p(\text{shiny}|\text{wet})$
- $p(\text{not shiny}|\text{wet})$
- $p(\text{shiny}|\text{dry})$
- $p(\text{not shiny}|\text{dry})$

In [ ]:
likelihoods = {
    'wet': {
        'shiny': ...,
        'not_shiny': ...
    },
    'dry': {
        'shiny': ...,
        'not_shiny': ...
    }
}

## Step 2: Inference

Now that we have defined the alterative hypotheses on the possible states of the floor (_wet_ or _dry_), your priors for each of them, and the likelihoods of each possible observation under each hypothesis, let's perform **inference**.  

Inference is done when we make an **observation**, and we wonder what world state it is the result of.

> **_Observation:_** Suppose that the observer **sees the floor as shiny**.

What can they infer on the actual **state** of the floor (i.e. the probability of it actually being wet)?

<img src="images/inference.png" style="height: 150px; margin-left: 10px;" />

We use **Bayes' rule** as a formula to update the prior belief and take into consideration the observation:

$$
\underbrace{P(H|\text{obs})}_\text{posterior} = \frac{\overbrace{p(\text{obs|H})}^\text{likelihood}\overbrace{p(H)}^\text{prior}}{p(\text{obs})}
$$

Let's break this down with our example. Let's say that we want to evaluate the probability that the floor is _wet_, given that we see it _shiny_.  
- In other words, we are looking for $p(\text{wet}|\text{shiny})$, the **posterior** that is informed by the obsevation.
- This probability is proportional to the **likelihood** of any wet floor to be perceived as shiny, $P(\text{shiny}|\text{wet})$, modulated by the how often we encounter wet floors in general $p(\text{wet})$ (our **prior**).
- This numerator is equivalent to the probability of seeing a _wet_ AND _shiny_ floor (also known as _joint probability_)
- To get the posterior, we need to divide the joint probability by the probability of seeing a shiny floor in general $p(\text{shiny})$ (**normalizing constant**)

The denominator $p(\text{shiny})$ represents the probability of seeing a shiny floor in all possible world states. This piece of information isn't given to you, but I let you think about how you can calculate it. Don't hesitate to discuss it with your neighbor if you haven't done this before.

📝 Complete the `bayesian_inference` function below. It takes as an input:
- An `observation` (as a string: 'shiny' or 'not_shiny')
- the `priors` (the dictionary you defined above)
- the `likelihoods` (the dictionary you defined above)

It then computes and returns the `posteriors` dictionary. The latter has exactly the same format as the priors, except that the probability values will have changed. 

In [ ]:
def bayesian_inference(observation: str, priors: dict, likelihoods: dict) -> dict:
    posteriors = {}   # Create the output dictionary
    hypotheses = priors.keys()  # The keys of the priors dictionary are ['wet', 'dry']

    # Compute a posterior probability for each hypothesis
    for hypothesis in hypotheses:
        # Your code here
        ...

        posteriors[hypothesis] = ...
        
    return posteriors

Imagine that you **observe** a _shiny_ floor, just as in the example we gave. 

📝 Call your `bayesian_inference` function with the hypothesis that you want to evaluate. You have unit tests to check that your implementation is correct.

In [ ]:
observation = ...

posteriors = bayesian_inference(..., ..., ...)
print(posteriors)

# Unit tests
assert isinstance(posteriors, dict), "Your posterior should be a dictionary"
assert set(posteriors.keys()) == {'wet', 'dry'}, "Your posteriors dictionary should have the same format as the priors."
assert math.isclose(posteriors['wet'], 0.182, abs_tol=0.001), "Incorrect posterior value for the 'wet' hypothesis"
assert math.isclose(posteriors['dry'], 0.818, abs_tol=0.001), "Incorrect posterior value for the 'dry' hypothesis"
print("OK 👌")

📝 Call your `bayesian_inference` function with the other observation and see how the posterior changes

📝 Change the prior and see how it impacts the inference.

In [ ]:
new_prior = {
    ...
}
observation = ...

bayesian_inference(..., ..., ...)

### References: 
1. Ma, W. J., Kording, K. P., & Goldreich, D. (2023). Bayesian Models of Perception and Action: An Introduction. MIT press.  
Available https://www.cns.nyu.edu/malab/bayesianbook.html
